# MCP — Agent-Analyse

Tier-1 eval (#384). Tests MCP-tool detection + native-only enforcement.
Default ohne MCP-Server (native_only-Szenarien); Sektion 7 für optionalen
document-extraction-mcp Live-Run.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)
print('factory + executor ready')

## 2. Agent bauen

Default: **kein** MCP-Server konfiguriert → native_only-Tests sollten passen.
Im optionalen Abschnitt 7 kann document-extraction-mcp aktiviert werden.

In [ ]:
WORKDIR = Path('.taskforce_mcp_analysis')
WORKDIR.mkdir(exist_ok=True)
BASE_TOOLS = ['web_search', 'web_fetch', 'file_read', 'python']
MCP_SERVERS: list[dict] = []   # populate in section 7 to test MCP path

from taskforce.infrastructure.tools.mcp.wrapper import MCPToolWrapper

def _mcp_tool_names_of(agent) -> set[str]:
    out = set()
    for name, tool in agent.tools.items():
        target = getattr(tool, '_original', tool)
        if isinstance(target, MCPToolWrapper):
            out.add(name)
    return out

async def build_mcp_agent():
    a = await factory.create_agent(
        system_prompt='Du bist ein knapper Assistent. Antworte direkt.',
        tools=BASE_TOOLS,
        mcp_servers=MCP_SERVERS or None,
        persistence={'type': 'file', 'work_dir': str(WORKDIR)},
        work_dir=str(WORKDIR),
        planning_strategy='native_react', max_steps=6,
    )
    alib.patch_anti_compression(a)
    return a, 50

BUILD_AGENT = build_mcp_agent
_smoke, _chars = await BUILD_AGENT()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
MCP_TOOL_NAMES = _mcp_tool_names_of(_smoke)
print(f'all tools : {sorted(AVAILABLE_TOOLS)}')
print(f'mcp tools : {sorted(MCP_TOOL_NAMES) or "(none)"}')
await _smoke.close()

## 3. Szenarien laden

Aus `scenarios/mcp.yaml`.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/mcp.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)
print(f'Total: {len(all_scenarios)}, Eligible: {len(eligible)}')
for s in eligible:
    print(f'  - {s.id:35s} ({s.category}/{s.difficulty})')

## 4. Einzelnes Szenario (Detail-Lauf)

In [ ]:
TARGET_ID = 'mcp-native-only-simple'  # change me
target = next((s for s in eligible if s.id == TARGET_ID), None) or eligible[0]
print(f'Target: {target.id}')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')

In [ ]:
agent, sys_chars = await BUILD_AGENT()
rec = await alib.run(
    executor, agent, target.mission,
    project_root=WORKDIR, snapshot_subdirs=(),
    initial_system_prompt_chars=sys_chars, silent=False,
)
# Stash MCP tool names if any (no-op for non-MCP notebooks)
rec.extra['mcp_tool_names'] = list(_mcp_tool_names_of(agent)) if '_mcp_tool_names_of' in dir() else []
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_tool_results(rec, head=15)

In [ ]:
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based ({"PASS" if card.passed else "FAIL"}) ===')
alib.print_feature_checks(card)
if card.details:
    print('Details:')
    for d in card.details:
        print(f'  - {d}')

In [ ]:
await agent.close()
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')

## 6. Batch-Lauf

In [ ]:
results = await alib.run_scenarios(
    executor, BUILD_AGENT, eligible,
    project_root=WORKDIR, snapshot_subdirs=(),
    reset_dirs_before_each=(),
    repeats=1, progress=True,
)  
# Stash MCP tool names per-record so the mcp evaluator works
for r in results:
    r.record.extra['mcp_tool_names'] = list(MCP_TOOL_NAMES)
    # Re-score with the now-populated extra
    r.rule_score = alib.score_rule_based(r.record, r.scenario)

print()
alib.print_scenario_summary(results)

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Tool calls')

## 7. (Optional) document-extraction-mcp aktivieren

Setze `MCP_SERVERS = [...]` in Zelle 2 und re-run alle vorherigen Zellen,
um die `mcp-document-extract` Szenarien live zu testen.

```python
MCP_SERVERS = [{
    'type': 'stdio',
    'command': 'uv',
    'args': ['--directory', 'servers/document-extraction-mcp',
             'run', 'python', '-m', 'document_extraction_mcp.server'],
    'description': 'OCR + layout extraction',
}]
```

Schwere Deps (PyTorch, PaddleOCR) — nur lokal aktivieren.

## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')